# ⚙️ Run order (read first)

This notebook has interdependent steps. Run them in this order, and **re-run the build whenever you change detection/label logic** — the plugin reads data that the build produces.

1. **Sections 0–3** — environment, load nuScenes, GPS sanity-check, Mapbox token
2. **Section 4** — build the grouped dataset *(computes camera cuboids, lidar boxes, per-keyframe detection lat/lons, clean labels)*
3. **Section 4b** — verify the build populated `scene_detections`
4. **Sections 5 / 5b** — spatial index + crosswalk/lane overlays
5. **Section 5c** — convert lidar → `.pcd`/`.fo3d` + orthographic thumbnails (so the App carousel shows previews)
6. **Section 6** — launch with the built-in Map panel (works without the custom plugin)
7. **Section 7** — write/build/install the custom **Frame + Map** panel, then **restart the kernel** and relaunch

**Golden rule:** if you change anything in Section 4, re-run Section 4 **and** 4b before expecting the panel to reflect it. The plugin only shows what the build stored.


# nuScenes → Grouped Dataset (6 cameras + LIDAR) with a Trajectory Map Panel

**Goal:** load a few nuScenes **mini** scenes as a **grouped dataset** — each keyframe is a group with all 6 cameras + the LIDAR point cloud as slices — with a **GPS location** on every group. Then build a custom panel that draws the scene's full **trajectory** on a Mapbox map and **auto-advances an animated marker** through the keyframes, alongside the camera frame (with projected 3D boxes) and a bird's-eye LIDAR view — all stepping in sync.

### Scope notes (read first)
- **nuScenes "video" is keyframes, not playable video.** Cameras are loaded at the annotated keyframe rate (~2 Hz), so there is no `<video>` to scrub; a scene is a sequence of ~40 keyframe groups. The custom panel animates a marker along the route and advances the keyframe.
- **GPS is built-in.** The nuScenes devkit ships `derive_latlon`, which converts `ego_pose` + scene location to real lat/lon — no manual coordinate math.
- **Staged build.** Sections 1–6 get you a working grouped dataset plus the built-in Map panel (already a strong demo: 6 cameras + lidar + map). Section 7 adds the custom synced Frame + Map panel.

### Assumptions
- **macOS** (Apple Silicon or Intel) with a working **Python virtual environment** (examples assume one at `~/fiftyone-plugins-dev/.venv`, but any venv works).
- **FiftyOne 1.17** installed in that venv.
- nuScenes **v1.0-mini** downloaded and unpacked (plus the Map expansion for crosswalks/lanes).
- **Node.js + npm** available (only needed for the custom JavaScript panel in Section 7).
- A free **Mapbox public access token** (`pk....`) for the map.

> The plugin is published under the namespace `@example/nuscenes-trajectory`. Rename it to your own namespace (e.g. `@yourname/...`) if you plan to share it.


## 0. Install dependencies

Activate your virtual environment and install the requirements:

```bash
# activate your venv (adjust the path to wherever yours lives)
source ~/fiftyone-plugins-dev/.venv/bin/activate

# core deps
pip install fiftyone==1.17 nuscenes-devkit 'numpy<2' jupyter

# for the LIDAR -> .pcd/.fo3d conversion and orthographic thumbnails (Section 5c)
pip install open3d
```

Notes:
- `nuscenes-devkit` works best with older NumPy, so `numpy<2` avoids conflicts.
- `open3d` provides Python 3.10–3.12 / Apple-Silicon wheels and is only needed for the LIDAR point-cloud conversion.


In [ ]:
import sys, shutil
import fiftyone as fo
print("python  :", sys.executable)
print("fiftyone:", fo.__version__)
try:
    import nuscenes; print("nuscenes-devkit: OK")
except Exception as e:
    print("nuscenes-devkit MISSING → pip install nuscenes-devkit 'numpy<2'", e)
node = shutil.which("node"); npm = shutil.which("npm")
print("node/npm:", bool(node and npm), "(needed only for the custom panel, Section 7)")

## 1. Point at the nuScenes mini download

Download **v1.0-mini** from https://www.nuscenes.org/ and unpack it (along with the Map expansion). Point the `dataroot` below at wherever you put it. A typical layout:

```
~/datasets/nuscenes/
  samples/   sweeps/   maps/   v1.0-mini/   (JSON metadata, incl. ego_pose.json)
```

If you keep the data elsewhere, just edit `NUSC_ROOT` in the next cell.


In [ ]:
from pathlib import Path
# >>> EDIT THIS if your nuScenes data lives elsewhere <<<
NUSC_ROOT = Path.home() / "datasets" / "nuscenes"
assert (NUSC_ROOT / "v1.0-mini").exists(), (
    f"Expected metadata at {NUSC_ROOT/'v1.0-mini'}.\n"
    "Unpack v1.0-mini.tgz here:  tar -xf v1.0-mini.tgz -C ~/datasets/nuscenes"
)
print("nuScenes root OK:", NUSC_ROOT)
print("contents:", [p.name for p in NUSC_ROOT.iterdir()])

In [ ]:
from nuscenes.nuscenes import NuScenes
nusc = NuScenes(version="v1.0-mini", dataroot=str(NUSC_ROOT), verbose=True)
print("\nscenes available:", len(nusc.scene))

## 2. Sanity-check the GPS conversion

`derive_latlon(location, [ego_pose,...])` returns lat/lon. We confirm it works on the first scene before building the whole dataset.

In [ ]:
from nuscenes.scripts.export_poses import derive_latlon

def ego_latlon(scene, sample_data_token):
    data = nusc.get("sample_data", sample_data_token)
    ego = nusc.get("ego_pose", data["ego_pose_token"])
    log = nusc.get("log", scene["log_token"])
    latlon = derive_latlon(log["location"], [ego])
    return latlon[0]["longitude"], latlon[0]["latitude"]

_sc = nusc.scene[0]
_first = nusc.get("sample", _sc["first_sample_token"])
_lidar_tok = _first["data"]["LIDAR_TOP"]
lon, lat = ego_latlon(_sc, _lidar_tok)
print(f"scene {_sc['name']} @ {nusc.get('log', _sc['log_token'])['location']}")
print(f"first keyframe lon/lat: {lon:.6f}, {lat:.6f}")

## 3. Mapbox token

In [ ]:
import os, getpass
token = os.environ.get("MAPBOX_TOKEN","").strip()
if not token.startswith("pk."): token = getpass.getpass("Paste Mapbox public token (pk....): ").strip()
assert token.startswith("pk."), "Need public token (pk.) from https://account.mapbox.com/access-tokens/"
os.environ["MAPBOX_TOKEN"] = token
fo.app_config.plugins = getattr(fo.app_config,"plugins",{}) or {}
fo.app_config.plugins["map"] = {"mapboxAccessToken": token}
print("token set:", token[:8], "…")

## 4. Build the grouped dataset (6 cameras + LIDAR, 3 scenes) + detections

Each **keyframe** becomes a group with the 6 cameras (images) + LIDAR (point cloud) as slices, every slice carrying the same `location` GeoLocation + the scene `trajectory` (JSON string).

**Detections, all classes:**
- **Cameras:** nuScenes 3D boxes are projected into each camera and stored as image-plane **cuboids** (`fo.Polylines` via `from_cuboid`). Boxes not visible in a given camera are skipped.
- **LIDAR:** the same 3D boxes are stored as 3D **`fo.Detections`** (location / dimensions / yaw) so they render as cuboids in the point-cloud viewer.

Labels are simplified (e.g. `vehicle.car` → `car`, `human.pedestrian.adult` → `pedestrian`). This cell is heavier than before because it projects boxes for every camera of every keyframe.

In [ ]:
import json
import numpy as np
from nuscenes.utils.geometry_utils import box_in_image, view_points, BoxVisibility
from pyquaternion import Quaternion
import os

# Derived-file dirs for converted lidar (.pcd.bin -> .pcd -> .fo3d)
PCD_DIR = Path.home() / "datasets" / "nuscenes_derived" / "pcd"
FO3D_DIR = Path.home() / "datasets" / "nuscenes_derived" / "fo3d"
for _d in (PCD_DIR, FO3D_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# Clear any stale derived files from earlier runs (e.g. malformed PCDs) so they regenerate.
import shutil as _shutil
for _d in (PCD_DIR, FO3D_DIR):
    for _f in _d.iterdir():
        try:
            _f.unlink()
        except Exception:
            pass

def bin_to_pcd(bin_path, pcd_path):
    """nuScenes LIDAR .pcd.bin (float32 x,y,z,intensity,ring) -> .pcd via open3d.

    We let open3d write the PCD so its own reader (used by FiftyOne's projection
    util) can always parse it back — a hand-written ASCII header is too easy to
    get subtly wrong for open3d's strict parser.
    """
    import open3d as o3d
    pts = np.fromfile(str(bin_path), dtype=np.float32).reshape(-1, 5)[:, :3]  # x,y,z
    pc = o3d.geometry.PointCloud()
    pc.points = o3d.utility.Vector3dVector(pts.astype(np.float64))
    o3d.io.write_point_cloud(str(pcd_path), pc)

def lidar_fo3d(bin_filepath):
    """Convert a nuScenes lidar .pcd.bin to a .fo3d scene; return the .fo3d path."""
    stem = Path(bin_filepath).stem.replace(".pcd", "")
    pcd_path = PCD_DIR / (stem + ".pcd")
    fo3d_path = FO3D_DIR / (stem + ".fo3d")
    if not pcd_path.exists():
        bin_to_pcd(bin_filepath, pcd_path)
    if not fo3d_path.exists():
        scene = fo.Scene()
        scene.add(fo.PointCloud("lidar", str(pcd_path)))
        scene.write(str(fo3d_path))
    return str(fo3d_path)

CAMERAS = ["CAM_FRONT", "CAM_FRONT_RIGHT", "CAM_BACK_RIGHT",
           "CAM_BACK", "CAM_BACK_LEFT", "CAM_FRONT_LEFT"]
LIDAR = "LIDAR_TOP"
N_SCENES = 3

# Which object classes to plot on the MAP (keeps the dot cloud readable).
# Set to None to include ALL classes. Barriers/cones are very numerous in nuScenes,
# so by default we focus the map on the dynamic agents.
PLOT_CLASSES = {"car", "truck", "bus", "trailer", "construction_vehicle",
                "motorcycle", "bicycle", "pedestrian"}

DATASET_NAME = "nuscenes-grouped-map"
if DATASET_NAME in fo.list_datasets():
    fo.delete_dataset(DATASET_NAME)
dataset = fo.Dataset(DATASET_NAME); dataset.persistent = True
dataset.add_group_field("group", default="CAM_FRONT")

def scene_trajectory(scene):
    pts = []
    tok = scene["first_sample_token"]
    while tok:
        s = nusc.get("sample", tok)
        lon, lat = ego_latlon(scene, s["data"][LIDAR])
        pts.append([lon, lat])
        tok = s["next"]
    return pts

_LABEL_MAP = {
    "vehicle.car": "car",
    "vehicle.truck": "truck",
    "vehicle.bus.bendy": "bus",
    "vehicle.bus.rigid": "bus",
    "vehicle.construction": "construction_vehicle",
    "vehicle.emergency.ambulance": "emergency",
    "vehicle.emergency.police": "emergency",
    "vehicle.trailer": "trailer",
    "vehicle.motorcycle": "motorcycle",
    "vehicle.bicycle": "bicycle",
    "human.pedestrian.adult": "pedestrian",
    "human.pedestrian.child": "pedestrian",
    "human.pedestrian.construction_worker": "pedestrian",
    "human.pedestrian.police_officer": "pedestrian",
    "human.pedestrian.personal_mobility": "pedestrian",
    "human.pedestrian.stroller": "pedestrian",
    "human.pedestrian.wheelchair": "pedestrian",
    "movable_object.barrier": "barrier",
    "movable_object.trafficcone": "traffic_cone",
    "movable_object.pushable_pullable": "movable_object",
    "movable_object.debris": "debris",
    "static_object.bicycle_rack": "bicycle_rack",
    "animal": "animal",
}

def category_to_label(name):
    if name in _LABEL_MAP:
        return _LABEL_MAP[name]
    return name.split(".")[-1]  # fallback: last dotted segment

def camera_cuboids(cam_token):
    """Project 3D boxes visible in this camera into image-plane cuboids (fo.Polyline)."""
    data_path, boxes, K = nusc.get_sample_data(cam_token, box_vis_level=BoxVisibility.ANY)
    from PIL import Image
    with Image.open(data_path) as im:
        W, H = im.size
    polylines = []
    for box in boxes:
        if not box_in_image(box, K, (H, W), vis_level=BoxVisibility.ANY):
            continue
        # 8 corners projected to 2D, normalized to [0,1]
        corners = view_points(box.corners(), K, normalize=True)[:2, :]  # 2x8
        corners[0, :] /= W
        corners[1, :] /= H
        c = corners.T.tolist()  # 8 x [x,y]
        label = category_to_label(box.name)
        try:
            pl = fo.Polyline.from_cuboid(c, label=label)
        except Exception:
            # fallback: 2D bbox from projected corner extents
            xs = [p[0] for p in c]; ys = [p[1] for p in c]
            x0, y0 = max(0, min(xs)), max(0, min(ys))
            x1, y1 = min(1, max(xs)), min(1, max(ys))
            pl = None
            polylines.append(("det", fo.Detection(label=label, bounding_box=[x0, y0, x1-x0, y1-y0])))
            continue
        polylines.append(("poly", pl))
    return polylines

def lidar_detections(lidar_token):
    """3D boxes in the LIDAR/ego frame as fo.Detections (location, dimensions, rotation)."""
    data_path, boxes, _ = nusc.get_sample_data(lidar_token, box_vis_level=BoxVisibility.NONE)
    dets = []
    for box in boxes:
        x, y, z = box.center.tolist()
        w, l, h = box.wlh.tolist()
        # yaw from quaternion -> [0,0,yaw] euler for FiftyOne 3D box
        yaw = box.orientation.yaw_pitch_roll[0]
        dets.append(fo.Detection(
            label=category_to_label(box.name),
            location=[x, y, z],
            dimensions=[l, w, h],
            rotation=[0, 0, float(yaw)],
        ))
    return dets

def keyframe_detections_latlon(scene, sample):
    """[{label, lng, lat}] for every annotated object in this keyframe (global->latlon)."""
    location = nusc.get("log", scene["log_token"])["location"]
    egos = []
    labels = []
    for ann_tok in sample["anns"]:
        ann = nusc.get("sample_annotation", ann_tok)
        lab = category_to_label(ann["category_name"])
        if PLOT_CLASSES is not None and lab not in PLOT_CLASSES:
            continue
        x, y, z = ann["translation"]  # GLOBAL frame already
        # derive_latlon expects ego_pose-like dicts with translation + timestamp
        egos.append({"translation": [float(x), float(y), float(z)],
                     "timestamp": sample["timestamp"]})
        labels.append(lab)
    if not egos:
        return []
    ll = derive_latlon(location, egos)
    out = []
    for lab, d in zip(labels, ll):
        out.append({"label": lab, "lng": d["longitude"], "lat": d["latitude"]})
    return out

# scene -> {keyframe -> [det,...]} collected during the build
scene_dets = {}

samples = []
for si in range(min(N_SCENES, len(nusc.scene))):
    scene = nusc.scene[si]
    traj = scene_trajectory(scene)
    traj_json = json.dumps(traj)
    tok = scene["first_sample_token"]
    kf = 0
    while tok:
        s = nusc.get("sample", tok)
        group = fo.Group()
        lon, lat = ego_latlon(scene, s["data"][LIDAR])
        loc = fo.GeoLocation(point=[lon, lat])
        kf_dets = keyframe_detections_latlon(scene, s)
        kf_dets_json = json.dumps(kf_dets)
        scene_dets.setdefault(scene["name"], {})[kf] = kf_dets

        for cam in CAMERAS:
            sd = nusc.get("sample_data", s["data"][cam])
            fp = str(NUSC_ROOT / sd["filename"])
            smp = fo.Sample(filepath=fp, group=group.element(cam))
            smp["location"] = loc
            smp["scene"] = scene["name"]
            smp["keyframe"] = kf
            smp["trajectory"] = traj_json
            # projected 3D->2D cuboids (and any bbox fallbacks)
            polys, dets = [], []
            for kind, obj in camera_cuboids(s["data"][cam]):
                (polys if kind == "poly" else dets).append(obj)
            if polys:
                smp["cuboids"] = fo.Polylines(polylines=polys)
            if dets:
                smp["detections"] = fo.Detections(detections=dets)
            samples.append(smp)

        sd = nusc.get("sample_data", s["data"][LIDAR])
        lp = str(NUSC_ROOT / sd["filename"])
        lp_fo3d = lidar_fo3d(lp)   # convert .pcd.bin -> .pcd -> .fo3d at build time
        smp = fo.Sample(filepath=lp_fo3d, group=group.element(LIDAR))
        smp["location"] = loc
        smp["scene"] = scene["name"]
        smp["keyframe"] = kf
        smp["trajectory"] = traj_json
        smp["det_points"] = kf_dets_json
        smp["detections"] = fo.Detections(detections=lidar_detections(s["data"][LIDAR]))
        samples.append(smp)

        tok = s["next"]; kf += 1
    print(f"scene {scene['name']}: {kf} keyframes")

dataset.add_samples(samples)
info = dict(dataset.info or {})
info["scene_detections"] = json.dumps(scene_dets)
dataset.info = info
dataset.save()
print("stored per-keyframe detection lat/lons in dataset.info['scene_detections']")
print("\ngroup media types:", dataset.group_media_types)
print(dataset)


## 4b. Verify the build populated detections (run after Section 4)

Quick check that `scene_detections` got stored and the coordinates look like real lat/lon (small numbers near Singapore ~103/1.3 or Boston ~-71/42). If this prints EMPTY or huge coordinates, fix it before launching the panel.

In [ ]:
import json as _json
_info = dataset.info or {}
_sd = _info.get("scene_detections")
if not _sd:
    print("❌ scene_detections EMPTY — re-run Section 4 (the build) before the panel will show map dots")
else:
    _sd = _json.loads(_sd) if isinstance(_sd, str) else _sd
    _scenes = list(_sd.keys())
    _k = _scenes[0]
    _kf0 = _sd[_k].get("0") or _sd[_k].get(0) or []
    _total = sum(len(v) for v in _sd[_k].values())
    print(f"✅ scenes: {_scenes}")
    print(f"   {_k}: {_total} detections across {len(_sd[_k])} keyframes")
    if _kf0:
        d = _kf0[0]
        ok = (-180 <= d["lng"] <= 180) and (-90 <= d["lat"] <= 90)
        print(f"   sample det: {d}")
        print("   coords look like lat/lon:" , "✅" if ok else "❌ (looks like raw meters — conversion failed)")

### Note on the LIDAR slice
nuScenes lidar is `.pcd.bin` (raw binary), which the FiftyOne 3D viewer may not render directly. If the lidar slice shows blank, it's a format issue, not a data issue — the cameras + map are the core of the demo. Converting `.pcd.bin` → `.pcd` is an optional enhancement (see the final notes cell).

## 5. Index location + confirm the group structure

In [ ]:
dataset.create_index("location.point", unique=False)
print("slices:", dataset.group_slices)
print("num groups:", len(dataset) // (len(CAMERAS)+1))
s = dataset.first()
print("sample scene/keyframe:", s["scene"], s["keyframe"])
print("location:", s["location"].point)
print("traj length:", len(json.loads(s["trajectory"])))

## 5b. Crosswalks + lanes → GeoJSON for the map panel

Reads the map expansion for each scene's city, grabs **ped_crossing** and **lane** polygons near the route, converts their local map coordinates to lat/lon with `derive_latlon`, and stores the result as a JSON string in `dataset.info` keyed by scene. The panel draws these as overlays so crosswalks/lanes appear on the Mapbox map.

Only polygons within a bounding box around the trajectory are kept — a whole-city dump would be enormous.

In [ ]:
from nuscenes.map_expansion.map_api import NuScenesMap
from nuscenes.scripts.export_poses import derive_latlon

# scene log location -> map name
def scene_map_name(scene):
    return nusc.get("log", scene["log_token"])["location"]

def _poly_latlon(nmap, location, polygon_token):
    """Return [[lng,lat],...] for a polygon record's exterior, converted to GPS."""
    poly = nmap.extract_polygon(polygon_token)  # shapely Polygon in map meters
    xs, ys = poly.exterior.coords.xy
    # derive_latlon expects ego-pose-like dicts with 'translation' [x,y,z]
    egos = [{"translation": [float(x), float(y), 0.0], "timestamp": 0} for x, y in zip(xs, ys)]
    ll = derive_latlon(location, egos)
    return [[d["longitude"], d["latitude"]] for d in ll]

def scene_map_overlays(scene, traj, layers=("ped_crossing", "lane"), pad=0.0015):
    """Crosswalk/lane polygons near the route, as GeoJSON-ish dict of [lng,lat] rings."""
    location = scene_map_name(scene)
    nmap = NuScenesMap(dataroot=str(NUSC_ROOT), map_name=location)
    # bbox around the trajectory (lat/lon) to limit how much we convert
    lons = [p[0] for p in traj]; lats = [p[1] for p in traj]
    box = (min(lons)-pad, min(lats)-pad, max(lons)+pad, max(lats)+pad)

    out = {"ped_crossing": [], "lane": []}
    for layer in layers:
        for rec in getattr(nmap, layer):
            tok = rec.get("polygon_token")
            if not tok:
                continue
            try:
                ring = _poly_latlon(nmap, location, tok)
            except Exception:
                continue
            # keep only polygons whose centroid falls in the route bbox
            cx = sum(p[0] for p in ring)/len(ring); cy = sum(p[1] for p in ring)/len(ring)
            if box[0] <= cx <= box[2] and box[1] <= cy <= box[3]:
                out[layer].append(ring)
    return out

# Build overlays per scene and stash in dataset.info (JSON string keyed by scene name)
overlays = {}
for si in range(min(N_SCENES, len(nusc.scene))):
    scene = nusc.scene[si]
    traj = scene_trajectory(scene)
    try:
        ov = scene_map_overlays(scene, traj)
        overlays[scene["name"]] = ov
        print(f"{scene['name']}: {len(ov['ped_crossing'])} crosswalks, {len(ov['lane'])} lanes near route")
    except Exception as e:
        print(f"{scene['name']}: map overlay failed: {e}")

info = dict(dataset.info or {})
info["map_overlays"] = json.dumps(overlays)
dataset.info = info
dataset.save()
print("stored map overlays in dataset.info['map_overlays']")

import fiftyone.utils.utils3d as fou3d

# Lidar samples are already .fo3d (converted at build time in Section 4).
# This computes the bird's-eye orthographic thumbnails so the carousel shows a
# preview instead of a broken icon. It REQUIRES open3d:
#     pip install open3d
# open3d wheels can lag on the newest Python on Apple Silicon; if it isn't available,
# this cell skips thumbnail generation gracefully (the panel BEV + map still work).
PROJ_DIR = Path.home() / "datasets" / "nuscenes_derived" / "proj"
PROJ_DIR.mkdir(parents=True, exist_ok=True)

try:
    import open3d  # noqa: F401
    _has_o3d = True
except Exception:
    _has_o3d = False

if not _has_o3d:
    print("⚠️ open3d not installed — skipping carousel thumbnails.")
    print("   Install it to enable previews:  pip install open3d")
    print("   (The panel's bird's-eye view + map detections work without this.)")
else:
    lidar_view = dataset.select_group_slices(LIDAR)
    # skip_failures=True so one bad cloud can't abort the whole run
    fou3d.compute_orthographic_projection_images(
        lidar_view, (-1, 512), str(PROJ_DIR), skip_failures=True
    )
    print("✅ computed orthographic projection thumbnails — carousel will show BEV previews")


In [ ]:
import os
import numpy as np
import fiftyone.utils.utils3d as fou3d

PCD_DIR = Path.home() / "datasets" / "nuscenes_derived" / "pcd"
FO3D_DIR = Path.home() / "datasets" / "nuscenes_derived" / "fo3d"
PROJ_DIR = Path.home() / "datasets" / "nuscenes_derived" / "proj"
for d in (PCD_DIR, FO3D_DIR, PROJ_DIR):
    d.mkdir(parents=True, exist_ok=True)

def bin_to_pcd(bin_path, pcd_path):
    """nuScenes LIDAR .pcd.bin (float32 x,y,z,intensity,ring) -> ASCII .pcd (x,y,z,intensity)."""
    pts = np.fromfile(str(bin_path), dtype=np.float32)
    pts = pts.reshape(-1, 5)[:, :4]  # x,y,z,intensity
    n = pts.shape[0]
    header = (
        "# .PCD v0.7 - Point Cloud Data file format\\n"
        "VERSION 0.7\\n"
        "FIELDS x y z intensity\\n"
        "SIZE 4 4 4 4\\n"
        "TYPE F F F F\\n"
        "COUNT 1 1 1 1\\n"
        f"WIDTH {n}\\n"
        "HEIGHT 1\\n"
        "VIEWPOINT 0 0 0 1 0 0 0\\n"
        f"POINTS {n}\\n"
        "DATA ascii\\n"
    )
    with open(pcd_path, "w") as f:
        f.write(header)
        for p in pts:
            f.write(f"{p[0]:.4f} {p[1]:.4f} {p[2]:.4f} {p[3]:.2f}\\n")

# Convert each LIDAR sample, write a .fo3d, and repoint the sample.
lidar_view = dataset.select_group_slices(LIDAR)
n_done = 0
for smp in lidar_view:
    binp = smp.filepath
    if not binp.endswith(".pcd.bin"):
        continue  # already converted
    stem = Path(binp).stem.replace(".pcd", "")  # drop .bin then .pcd
    pcd_path = PCD_DIR / (stem + ".pcd")
    fo3d_path = FO3D_DIR / (stem + ".fo3d")
    if not pcd_path.exists():
        bin_to_pcd(binp, pcd_path)
    # build a .fo3d scene referencing the .pcd
    scene = fo.Scene()
    scene.add(fo.PointCloud("lidar", str(pcd_path)))
    scene.write(str(fo3d_path))
    smp.filepath = str(fo3d_path)
    smp.save()
    n_done += 1
print(f"converted + repointed {n_done} LIDAR samples to .fo3d")

# Compute orthographic (bird's-eye) projection images for the lidar slice.
lidar_view = dataset.select_group_slices(LIDAR)
fou3d.compute_orthographic_projection_images(lidar_view, (-1, 512), str(PROJ_DIR))
print("computed orthographic projection thumbnails \u2014 carousel will now show BEV previews")

## 6. Launch the App — built-in Map panel beside the sensors (guaranteed demo)

Even before the custom panel, this is compelling:
1. The grid shows groups; switch the active slice (CAM_FRONT, CAM_BACK, …, LIDAR_TOP) with the slice selector.
2. Add the built-in **Map** panel (`+` menu). It plots every keyframe's GPS.
3. Selecting a point on the map highlights the corresponding keyframe group; the map stays put as you switch sensor slices because `location` is on every slice.

Use a **dynamic group view ordered by keyframe** so each scene plays as an ordered sequence.

In [ ]:
from fiftyone import ViewField as F
# Order groups within each scene by keyframe (nice for stepping through time)
view = dataset.group_by("scene", order_by="keyframe")
session = fo.launch_app(view, auto=False)
print("App URL:", session.url)
try: session.open_tab()
except Exception as e: print("open the URL above:", e)

## 7. Custom panel — frame + map, frames play in lockstep with the marker

Writes/builds/installs a JS panel + a Python operator (`resolve_scene_media`). The operator takes the active **group id** + slice and returns, **in keyframe order for the whole scene**, that slice's served image URL + GPS point, plus the trajectory.

The panel then:
- shows the current keyframe's image on the **left**, the Mapbox map on the **right**
- draws the full scene route
- on **Play**, advances a keyframe timer that steps **both the image and the map marker together** — so the left frame plays like video while the marker tracks the same point on the map
- is **slice-aware**: it requests whichever slice the App reports active, so switching CAM_FRONT → CAM_BACK in the carousel reloads the scene's frames for that camera

**Layout in the App:** the carousel is the App's own component and can't live inside a panel. To get "carousel above, frame+map below," use the App's split-pane control (icons at the top-right of the panel tabs) to put the **Sample** view on top and dock **Frame + Map** beneath it — rather than as a side-by-side top tab.

Needs node/npm. Token + classic-JSX fix baked in.

In [ ]:
import subprocess
PLUGIN_SRC = Path.home()/"fiftyone-plugins-dev"/"nuscenes-trajectory-plugin"
(PLUGIN_SRC/"src").mkdir(parents=True, exist_ok=True)

(PLUGIN_SRC/"fiftyone.yml").write_text('''name: "@example/nuscenes-trajectory"
type: plugin
version: 1.0.0
description: Side-by-side current-frame image + trajectory map for grouped nuScenes scenes.
fiftyone:
  version: ">=1.0"
operators:
  - resolve_scene_media
panels:
  - name: TrajectoryMap
    label: "Frame + Map"
''')

# Operator: given a group id + slice, find the whole SCENE that group belongs to,
# and return, in keyframe order: the served image URL for that slice at each keyframe,
# plus the trajectory. This lets the panel play the frames in lockstep with the marker.
(PLUGIN_SRC/"__init__.py").write_text('''import json
import urllib.parse as up
import fiftyone as fo
import fiftyone.operators as foo
from fiftyone import ViewField as F


def _media_url(filepath):
    return "/media?filepath=" + up.quote(filepath)


class ResolveSceneMedia(foo.Operator):
    @property
    def config(self):
        return foo.OperatorConfig(
            name="resolve_scene_media",
            label="Resolve scene media",
            unlisted=True,
        )

    def execute(self, ctx):
        group_id = ctx.params.get("group_id")
        want_slice = ctx.params.get("slice") or "CAM_FRONT"
        if not group_id:
            return {"error": "no group_id"}

        ds = ctx.dataset

        # Which scene does this group belong to? Read it off any slice of the group.
        try:
            group = ds.get_group(group_id)
        except Exception as e:
            return {"error": "get_group failed: %s" % e}

        slices = list(group.keys())
        chosen = want_slice if want_slice in group else (
            "CAM_FRONT" if "CAM_FRONT" in group else slices[0]
        )
        ref = group[chosen]
        scene = ref.get_field("scene")
        traj = ref.get_field("trajectory")
        if isinstance(traj, str):
            try:
                traj = json.loads(traj)
            except Exception:
                traj = []

        # Gather, in keyframe order, the chosen slice's sample for every group in this scene.
        # We select the chosen slice, filter to this scene, and sort by keyframe.
        try:
            sv = ds.select_group_slices(chosen)
        except Exception:
            # older API name fallback
            sv = ds.select_group_slice(chosen)
        sv = sv.match(F("scene") == scene).sort_by("keyframe")

        frames = []  # list of {keyframe, url, point, boxes:[{label, pts:[[x,y]..]}]}
        for s in sv:
            loc = s.get_field("location")
            pt = loc.point if (loc is not None and loc.point is not None) else None
            boxes = []
            cub = s.get_field("cuboids")
            if cub is not None and getattr(cub, "polylines", None):
                for pl in cub.polylines:
                    # flatten polyline points (list of paths) to a single point list
                    pts = []
                    for path in (pl.points or []):
                        for xy in path:
                            pts.append([float(xy[0]), float(xy[1])])
                    if pts:
                        boxes.append({"label": pl.label, "pts": pts})
            # bird's-eye detections from the LIDAR slice of the SAME group/keyframe
            bev = []
            try:
                gid_k = s.get_field("group")
                gid_k = (gid_k.id if hasattr(gid_k, "id") else None)
                if gid_k:
                    grp = ds.get_group(gid_k)
                    lidar_s = grp.get("LIDAR_TOP")
                    if lidar_s is not None:
                        ldets = lidar_s.get_field("detections")
                        if ldets is not None and getattr(ldets, "detections", None):
                            for d in ldets.detections:
                                locv = d.location or [0, 0, 0]
                                dimv = d.dimensions or [1, 1, 1]   # [l, w, h]
                                rotv = d.rotation or [0, 0, 0]     # [_, _, yaw]
                                bev.append({"x": float(locv[0]), "y": float(locv[1]),
                                            "l": float(dimv[0]), "w": float(dimv[1]),
                                            "yaw": float(rotv[2]), "label": d.label})
            except Exception:
                bev = []
            frames.append({
                "keyframe": s.get_field("keyframe"),
                "url": _media_url(s.filepath),
                "point": pt,
                "boxes": boxes,
                "bev": bev,
            })

        # crosswalk/lane overlays for this scene (stored in dataset.info as JSON)
        overlays = {}
        try:
            info = ds.info or {}
            allov = info.get("map_overlays")
            if isinstance(allov, str):
                allov = json.loads(allov)
            if isinstance(allov, dict):
                overlays = allov.get(scene, {}) or {}
        except Exception:
            overlays = {}

        # per-keyframe object detections (lat/lon) for this scene
        scene_dets = {}
        try:
            info = ds.info or {}
            alld = info.get("scene_detections")
            if isinstance(alld, str):
                alld = json.loads(alld)
            if isinstance(alld, dict):
                scene_dets = alld.get(scene, {}) or {}
        except Exception:
            scene_dets = {}

        return {
            "slice": chosen,
            "slices": slices,
            "scene": scene,
            "trajectory": traj or [],
            "frames": frames,   # ordered by keyframe
            "overlays": overlays,
            "detections": scene_dets,  # {keyframe(str/int): [{label,lng,lat}]}
        }


def register(p):
    p.register(ResolveSceneMedia)
''')

(PLUGIN_SRC/"package.json").write_text('''{
  "name": "@example/nuscenes-trajectory",
  "version": "1.0.0",
  "type": "module",
  "fiftyone": { "script": "dist/index.umd.js" },
  "scripts": { "build": "vite build" },
  "dependencies": { "mapbox-gl": "^3.0.0" },
  "devDependencies": { "@vitejs/plugin-react": "^4.0.0", "vite": "^5.0.0" }
}
''')

(PLUGIN_SRC/"vite.config.js").write_text('''import { defineConfig } from "vite";
import react from "@vitejs/plugin-react";
const EXTERNAL = ["react","react-dom","react/jsx-runtime","recoil","@fiftyone/plugins","@fiftyone/state","@fiftyone/operators","@fiftyone/components","@fiftyone/utilities"];
export default defineConfig({
  plugins: [react({ jsxRuntime: "classic" })],
  build: {
    lib: { entry: "src/index.jsx", name: "TrajectoryMap", formats: ["umd"], fileName: () => "index.umd.js" },
    rollupOptions: { external: EXTERNAL, output: { globals: {
      react:"React", "react-dom":"ReactDOM", "react/jsx-runtime":"jsxRuntime", recoil:"recoil",
      "@fiftyone/plugins":"__fop__", "@fiftyone/state":"__fos__", "@fiftyone/operators":"__foo__",
      "@fiftyone/components":"__foc__", "@fiftyone/utilities":"__fou__" } } },
  },
});
''')
print("wrote manifest (+scene-media operator), __init__.py, package.json, vite.config.js")


In [ ]:
panel_jsx = r"""import React, { useEffect, useRef, useState, useCallback } from "react";
import * as fop from "@fiftyone/plugins";
import * as fos from "@fiftyone/state";
import * as foo from "@fiftyone/operators";
import { useRecoilValue } from "recoil";
import mapboxgl from "mapbox-gl";
import "mapbox-gl/dist/mapbox-gl.css";

const MAPBOX_TOKEN = "__TOKEN__";
const FRAME_MS = 120;  // playback pace: ms per keyframe (~8 fps feel)

export function TrajectoryMap() {
  const mapEl = useRef(null), mapRef = useRef(null), markerRef = useRef(null);
  const timerRef = useRef(null), idxRef = useRef(0);
  const allDetsRef = useRef([]);
  const imgRef = useRef(null);
  const [imgBox, setImgBox] = useState({w:0,h:0});

  let sample = undefined;
  try { sample = useRecoilValue(fos.modalSample); } catch (e) {}

  // frames: ordered [{keyframe, url, point}]; pts: [lng,lat] per frame
  const [frames, setFrames] = useState([]);
  const [pts, setPts] = useState([]);
  const [traj, setTraj] = useState([]);
  const [overlays, setOverlays] = useState(null);
  const [sceneDets, setSceneDets] = useState(null);
  const [legend, setLegend] = useState([]);
  const [showAll, setShowAll] = useState(false);
  const [idx, setIdx] = useState(0);
  const [activeSlice, setActiveSlice] = useState(null);
  const [status, setStatus] = useState("open a group to begin");
  const [playing, setPlaying] = useState(false);

  const exec = foo.useOperatorExecutor("@example/nuscenes-trajectory/resolve_scene_media");

  useEffect(() => {
    const src = (sample && (sample.sample ?? sample)) || null;
    if (!src) { setStatus("no active group \u2014 open a sample in the modal"); return; }
    const gid = (src.group && (src.group._id || src.group.id)) || src.group_id;
    let slice = undefined;
    try { slice = src.group && src.group.name; } catch (e) {}
    if (!gid) { setStatus("no group id on sample"); return; }
    setStatus("loading scene frames\u2026 slice=" + (slice || "default"));
    try { exec.execute({ group_id: String(gid), slice: slice || null }); }
    catch (e) { setStatus("execute() threw: " + String(e)); }
  }, [sample]);

  useEffect(() => {
    if (!exec || !exec.result) return;
    const r = exec.result || {};
    if (r.error) { setStatus("operator: " + r.error); return; }
    const fr = Array.isArray(r.frames) ? r.frames : [];
    setFrames(fr);
    setPts(fr.map(function(x){ return x.point; }).filter(Boolean));
    setTraj(Array.isArray(r.trajectory) ? r.trajectory : []);
    setOverlays(r.overlays || null);
    setSceneDets(r.detections || null);
    setActiveSlice(r.slice);
    idxRef.current = 0; setIdx(0);
    setStatus(fr.length ? "" : "no frames returned");
  }, [exec && exec.result]);

  useEffect(() => {
    if (exec && exec.error) setStatus("operator error: " + String(exec.error));
  }, [exec && exec.error]);

  // init map
  useEffect(() => {
    if (!mapEl.current || mapRef.current) return;
    if (!MAPBOX_TOKEN.startsWith("pk.")) { setStatus("Mapbox token missing"); return; }
    mapboxgl.accessToken = MAPBOX_TOKEN;
    const map = new mapboxgl.Map({ container: mapEl.current, style:"mapbox://styles/mapbox/dark-v11", center:[-122,37.5], zoom:13 });
    mapRef.current = map;
    const el = document.createElement("div");
    el.style.cssText = "width:18px;height:18px;border-radius:50%;background:#ff6b35;border:3px solid #fff;box-shadow:0 0 0 3px rgba(255,107,53,.4)";
    markerRef.current = new mapboxgl.Marker({ element: el });
    // Mapbox measures its container on create; if the panel sized/docked after,
    // it paints into the wrong box. Force re-measure on load + whenever the
    // container resizes (docking, split-pane, tab switches).
    map.on("load", () => map.resize());
    let ro = null;
    try {
      ro = new ResizeObserver(() => { try { map.resize(); } catch(e){} });
      ro.observe(mapEl.current);
    } catch(e) {}
    // also nudge a few times right after mount to catch late layout
    const t1 = setTimeout(() => { try{map.resize();}catch(e){} }, 100);
    const t2 = setTimeout(() => { try{map.resize();}catch(e){} }, 400);
    const t3 = setTimeout(() => { try{map.resize();}catch(e){} }, 1000);
    return () => {
      clearTimeout(t1); clearTimeout(t2); clearTimeout(t3);
      if (ro) { try { ro.disconnect(); } catch(e){} }
      try{map.remove();}catch(e){} mapRef.current=null;
    };
  }, []);

  // draw route + fit when traj changes
  useEffect(() => {
    const map = mapRef.current; if (!map || !traj.length) return;
    const draw = () => {
      const line = { type:"Feature", geometry:{ type:"LineString", coordinates: traj } };
      if (map.getSource("r")) map.getSource("r").setData(line);
      else { map.addSource("r",{type:"geojson",data:line});
        map.addLayer({id:"r",type:"line",source:"r",paint:{"line-color":"#ff6b35","line-width":4,"line-opacity":0.9}}); }
      const b = traj.reduce((a,c)=>a.extend(c), new mapboxgl.LngLatBounds(traj[0],traj[0]));
      map.fitBounds(b,{padding:50,duration:0});
      if (pts.length) markerRef.current.setLngLat(pts[0]).addTo(map);
      else markerRef.current.setLngLat(traj[0]).addTo(map);
    };
    if (map.isStyleLoaded()) draw(); else map.once("load", draw);
  }, [traj, pts]);

  // draw crosswalk + lane polygons when overlays arrive
  useEffect(() => {
    const map = mapRef.current; if (!map || !overlays) return;
    const drawPolys = (id, rings, color, opacity) => {
      const fc = { type:"FeatureCollection", features: (rings||[]).map(function(r){
        return { type:"Feature", geometry:{ type:"Polygon", coordinates:[r] } };
      })};
      if (map.getSource(id)) { map.getSource(id).setData(fc); return; }
      map.addSource(id, { type:"geojson", data: fc });
      map.addLayer({ id:id+"-fill", type:"fill", source:id,
        paint:{ "fill-color":color, "fill-opacity":opacity } });
      map.addLayer({ id:id+"-line", type:"line", source:id,
        paint:{ "line-color":color, "line-width":1.2, "line-opacity":0.8 } });
    };
    const draw = () => {
      drawPolys("xwalk", overlays.ped_crossing, "#37a0ff", 0.45);
      drawPolys("lanes", overlays.lane, "#9b9b9b", 0.18);
    };
    if (map.isStyleLoaded()) draw(); else map.once("load", draw);
  }, [overlays]);

  // ONE shared class->color used by frame boxes, map dots, and the legend
  const DET_COLORS = ["#ff6b35","#37a0ff","#4caf50","#ffce42","#b86bff","#ff5d8f","#33d6c0","#e0e0e0","#ff8a3d","#7e9cff"];
  const detColor = (label) => {
    let h=0; for (let i=0;i<(label||"").length;i++) h=(h*31+label.charCodeAt(i))>>>0;
    return DET_COLORS[h % DET_COLORS.length];
  };
  const colorFor = detColor;  // alias so the frame overlay uses the same colors
  const toFC = (arr) => ({ type:"FeatureCollection", features:(arr||[]).map(function(d){
    return { type:"Feature", properties:{ label:d.label, color:detColor(d.label) },
             geometry:{ type:"Point", coordinates:[d.lng, d.lat] } };
  })});

  // draw the faint all-scene detection cloud + build legend when detections arrive
  useEffect(() => {
    const map = mapRef.current; if (!map || !sceneDets) return;
    const all = [];
    const labelsSeen = {};
    Object.keys(sceneDets).forEach(function(k){
      (sceneDets[k]||[]).forEach(function(d){ all.push(d); labelsSeen[d.label]=detColor(d.label); });
    });
    setLegend(Object.keys(labelsSeen).sort().map(function(l){ return {label:l, color:labelsSeen[l]}; }));
    const draw = () => {
      // store the full set so the toggle effect can build the faint layer on demand
      allDetsRef.current = all;
      // current-keyframe layer (the always-on bright dots)
      if (!map.getSource("dets-cur")) {
        map.addSource("dets-cur", { type:"geojson", data: toFC([]) });
        map.addLayer({ id:"dets-cur", type:"circle", source:"dets-cur",
          paint:{ "circle-radius":7, "circle-color":["get","color"],
                  "circle-stroke-width":2, "circle-stroke-color":"#fff", "circle-opacity":1 } });
      }
    };
    if (map.isStyleLoaded()) { draw(); }
    else {
      // 'load' may have already fired for late effects; retry on style readiness
      const onReady = () => { if (map.isStyleLoaded()) { draw(); map.off("styledata", onReady); map.off("idle", onReady); } };
      map.on("styledata", onReady);
      map.on("idle", onReady);
      const t = setTimeout(() => { try { draw(); } catch(e){} }, 600);
      // best-effort cleanup
      setTimeout(() => { map.off("styledata", onReady); map.off("idle", onReady); }, 5000);
    }
  }, [sceneDets]);

  // update the bright current-keyframe detections as idx changes
  useEffect(() => {
    const map = mapRef.current; if (!map || !sceneDets) return;
    const cur = sceneDets[String(idx)] || sceneDets[idx] || [];
    const set = () => { const s = map.getSource("dets-cur"); if (s) s.setData(toFC(cur)); };
    if (map.getSource("dets-cur")) set();
    else { const t = setTimeout(set, 700); }
  }, [idx, sceneDets]);

    // add/remove the faint all-scene cloud based on the toggle (no reliance on visibility state)
  useEffect(() => {
    const map = mapRef.current; if (!map) return;
    const apply = () => {
      const exists = !!map.getLayer("dets-all");
      if (showAll && !exists) {
        if (!map.getSource("dets-all")) map.addSource("dets-all", { type:"geojson", data: toFC(allDetsRef.current) });
        else map.getSource("dets-all").setData(toFC(allDetsRef.current));
        map.addLayer({ id:"dets-all", type:"circle", source:"dets-all",
          paint:{ "circle-radius":2, "circle-color":["get","color"], "circle-opacity":0.15 } },
          "dets-cur");  // insert BELOW the bright current layer
      } else if (!showAll && exists) {
        map.removeLayer("dets-all");
      }
    };
    if (map.isStyleLoaded()) apply(); else { const t = setTimeout(apply, 300); }
  }, [showAll, sceneDets]);

    // place marker at the current frame's point
  const placeMarker = useCallback((i) => {
    const map = mapRef.current;
    if (!markerRef.current || !map) return;
    const p = pts[i] || traj[Math.min(i, traj.length-1)];
    if (p) {
      markerRef.current.setLngLat(p);
      try { markerRef.current.addTo(map); } catch(e) {}  // ensure attached
    }
  }, [pts, traj]);

  // place the marker as soon as we have points (so it shows before Play)
  useEffect(() => {
    if (pts.length || traj.length) placeMarker(idxRef.current || 0);
  }, [pts, traj, placeMarker]);

    // playback: advance idx on an interval, stepping BOTH image and marker
  useEffect(() => {
    if (!playing) { if (timerRef.current) clearInterval(timerRef.current); return; }
    if (!frames.length) { setPlaying(false); return; }
    timerRef.current = setInterval(() => {
      let n = idxRef.current + 1;
      if (n >= frames.length) { n = frames.length - 1; setPlaying(false); }
      idxRef.current = n; setIdx(n); placeMarker(n);
    }, FRAME_MS);
    return () => { if (timerRef.current) clearInterval(timerRef.current); };
  }, [playing, frames, placeMarker]);

  const reset = () => { idxRef.current = 0; setIdx(0); placeMarker(0); setPlaying(false); };

  const curFrame = frames.length ? frames[Math.min(idx, frames.length-1)] : null;
  const curUrl = curFrame ? curFrame.url : null;
  const curBoxes = (curFrame && Array.isArray(curFrame.boxes)) ? curFrame.boxes : [];
  const curBev = (curFrame && Array.isArray(curFrame.bev)) ? curFrame.bev : [];
  const isLidar = (activeSlice === "LIDAR_TOP");
  // BEV: ego frame meters -> normalized [0,1], ego center, half-range BEV_RANGE meters.
  const BEV_RANGE = 60;
  const m2n = (meters) => meters / (2 * BEV_RANGE);     // meters -> normalized length
  const bevXY = (x, y) => {
    // x = forward (up on screen), y = left (left on screen)
    return [0.5 - (y / (2 * BEV_RANGE)), 0.5 - (x / (2 * BEV_RANGE))];
  };
  // build an SVG transform that places + rotates a box at (x,y) with yaw (radians)
  const bevBox = (d) => {
    const [cx, cy] = bevXY(d.x, d.y);
    const Ln = m2n(d.l || 1), Wn = m2n(d.w || 1);
    // yaw: 0 = facing +x (forward/up). SVG rotate is clockwise in degrees about center.
    // screen up is -y, forward is +x(world) -> rotate so the box length aligns to heading.
    const deg = -(d.yaw || 0) * 180 / Math.PI;
    return { cx, cy, Ln, Wn, deg };
  };

  // bounding box (in normalized coords) of a cuboid's points, for label placement
  const boxBounds = (pts) => {
    let xs = pts.map(p=>p[0]), ys = pts.map(p=>p[1]);
    return { x: Math.min.apply(null,xs), y: Math.min.apply(null,ys),
             X: Math.max.apply(null,xs), Y: Math.max.apply(null,ys) };
  };
  // (color helper defined once below as colorFor / detColor share it)

  // measure the rendered image so overlays can match its EXACT pixel box
  const measureImg = useCallback(() => {
    const el = imgRef.current; if (!el) return;
    const w = el.clientWidth, h = el.clientHeight;
    if (w && h) setImgBox(function(prev){ return (prev.w===w && prev.h===h) ? prev : {w:w, h:h}; });
  }, []);
  useEffect(() => {
    measureImg();
    let ro = null;
    try { ro = new ResizeObserver(measureImg); if (imgRef.current) ro.observe(imgRef.current); } catch(e){}
    window.addEventListener("resize", measureImg);
    return () => { if (ro) { try{ro.disconnect();}catch(e){} } window.removeEventListener("resize", measureImg); };
  }, [measureImg, curUrl]);

    return (
    <div style={{display:"flex",flexDirection:"column",width:"100%",height:"100%",overflow:"hidden"}}>
      <div style={{display:"flex",gap:8,alignItems:"center",padding:"6px 8px",flexWrap:"wrap",flex:"0 0 auto"}}>
        <button onClick={()=>setPlaying(true)} disabled={playing}
                style={{padding:"4px 12px",cursor:playing?"default":"pointer",opacity:playing?0.5:1}}>Play</button>
        <button onClick={()=>setPlaying(false)} disabled={!playing}
                style={{padding:"4px 12px",cursor:!playing?"default":"pointer",opacity:!playing?0.5:1}}>Pause</button>
        <button onClick={reset} style={{padding:"4px 12px",cursor:"pointer"}}>Reset</button>
        <span style={{fontSize:12,color:"#ddd"}}>{activeSlice ? ("slice: " + activeSlice) : ""}</span>
        <span style={{fontSize:12,color:"#bbb"}}>{status || ("frame " + (idx+1) + " / " + frames.length)}</span>
        <label style={{fontSize:11,color:"#aaa",display:"flex",alignItems:"center",gap:4,cursor:"pointer"}}>
          <input type="checkbox" checked={showAll} onChange={function(e){ setShowAll(e.target.checked); }} />
          show all-scene
        </label>
      </div>
      <div style={{display:"flex",flex:"1 1 0",minHeight:0,gap:8,overflow:"hidden",padding:"0 8px 8px"}}>
        
          {/* camera image (top) with cuboid overlay */}
          <div style={{flex:"1 1 0",minWidth:0,minHeight:0,overflow:"hidden",background:"#000",display:"flex",alignItems:"center",justifyContent:"center",position:"relative"}}>
            {curUrl
              ? <div style={{position:"relative", lineHeight:0}}>
                  <img ref={imgRef} src={curUrl} onLoad={measureImg}
                       onError={()=>setStatus("image failed: "+curUrl)}
                       style={{display:"block", maxWidth:"100%", maxHeight:"100%",
                               width:"auto", height:"auto"}} />
                  {/* overlay sized to the image's EXACT rendered pixels */}
                  <svg width={imgBox.w} height={imgBox.h} viewBox="0 0 1 1" preserveAspectRatio="none"
                       style={{position:"absolute", left:0, top:0, pointerEvents:"none"}}>
                    {curBoxes.map(function(b, bi){
                      const c = colorFor(b.label);
                      const ptsStr = b.pts.map(function(p){ return p[0]+","+p[1]; }).join(" ");
                      return <polyline key={bi} points={ptsStr} fill="none" stroke={c} strokeWidth={0.004}
                                vectorEffect="non-scaling-stroke" style={{strokeWidth:1.5}} />;
                    })}
                  </svg>
                  <div style={{position:"absolute", left:0, top:0, width:imgBox.w, height:imgBox.h, pointerEvents:"none"}}>
                    {curBoxes.map(function(b, bi){
                      const bb = boxBounds(b.pts); const c = colorFor(b.label);
                      const left = Math.max(0, Math.min(0.92, bb.x)) * 100;
                      const top = Math.max(0, Math.min(0.96, bb.y)) * 100;
                      return <div key={bi} style={{position:"absolute", left:left+"%", top:top+"%",
                          background:c, color:"#000", fontSize:9, lineHeight:"11px", fontWeight:700,
                          padding:"0 2px", whiteSpace:"nowrap", borderRadius:2,
                          boxShadow:"0 0 0 1px rgba(0,0,0,.4)"}}>{b.label}</div>;
                    })}
                  </div>
                </div>
              : <div style={{color:"#aaa",padding:16}}>{isLidar ? "LIDAR slice — see BEV below" : "No frame yet."}</div>}
          </div>
          {/* BEV (bottom) — always visible, synced lidar */}
          <div style={{flex:"1 1 0",minWidth:0,minHeight:0,background:"#07090c",display:"flex",alignItems:"center",justifyContent:"center"}}>
            <svg viewBox="0 0 1 1" preserveAspectRatio="xMidYMid meet" style={{width:"100%",height:"100%",maxWidth:"100%",maxHeight:"100%"}}>
              <rect x="0" y="0" width="1" height="1" fill="#07090c" />
              <line x1="0.5" y1="0.02" x2="0.5" y2="0.98" stroke="#1a2230" strokeWidth="0.0015" />
              <line x1="0.02" y1="0.5" x2="0.98" y2="0.5" stroke="#1a2230" strokeWidth="0.0015" />
              {[20,40,60].map(function(rm, ri){
                const rn = (rm / (2*BEV_RANGE));
                return <g key={ri}>
                  <circle cx="0.5" cy="0.5" r={rn} fill="none" stroke="#1f2a3a" strokeWidth="0.0015" />
                  <text x="0.5" y={0.5 - rn - 0.004} fill="#3a4a60" fontSize="0.022" textAnchor="middle">{rm}m</text>
                </g>;
              })}
              <polygon points="0.5,0.476 0.488,0.512 0.512,0.512" fill="#ffffff" />
              {curBev.map(function(d, di){
                const b = bevBox(d); const c = detColor(d.label);
                const x = b.cx - b.Wn/2, y = b.cy - b.Ln/2;
                return <g key={di} transform={"rotate(" + b.deg + " " + b.cx + " " + b.cy + ")"}>
                  <rect x={x} y={y} width={b.Wn} height={b.Ln} fill={c} fillOpacity="0.35" stroke={c} strokeWidth="0.0025" />
                  <line x1={b.cx} y1={b.cy} x2={b.cx} y2={y} stroke={c} strokeWidth="0.0025" />
                </g>;
              })}
              <text x="0.5" y="0.04" fill="#8aa0bd" fontSize="0.028" textAnchor="middle">LIDAR BEV — {curBev.length} objects (±{BEV_RANGE}m)</text>
            </svg>
          </div>
        
        <div style={{flex:"1.2 1 0",minWidth:0,minHeight:0,position:"relative"}}>
          <div ref={mapEl} style={{position:"absolute",inset:0}} />
          {legend.length ? <div style={{position:"absolute",top:6,right:6,background:"rgba(20,20,20,.8)",
              padding:"6px 8px",borderRadius:4,maxHeight:"45%",overflow:"auto",pointerEvents:"none"}}>
            {legend.map(function(it,li){ return <div key={li} style={{display:"flex",alignItems:"center",gap:6,fontSize:10,color:"#eee",lineHeight:"14px"}}>
              <span style={{width:9,height:9,borderRadius:"50%",background:it.color,display:"inline-block"}} />{it.label}</div>; })}
          </div> : null}
        </div>
      </div>
    </div>
  );
}

fop.registerComponent({
  name:"TrajectoryMap", label:"Frame + Map", component:TrajectoryMap,
  type: fop.PluginComponentType.Panel,
  panelOptions: { surfaces: "grid modal" },
});
"""
panel_jsx = panel_jsx.replace("__TOKEN__", token)
(PLUGIN_SRC/"src"/"index.jsx").write_text(panel_jsx)
print("wrote src/index.jsx (frames step with marker on Play)")

In [ ]:
if node and npm:
    print("npm install …"); r1=subprocess.run([npm,"install"],cwd=PLUGIN_SRC,capture_output=True,text=True); print(r1.stderr[-500:])
    print("npm run build …"); r2=subprocess.run([npm,"run","build"],cwd=PLUGIN_SRC,capture_output=True,text=True)
    print(r2.stdout[-400:]); print(r2.stderr[-1000:])
    print("BUILD OK" if (PLUGIN_SRC/"dist"/"index.umd.js").exists() else "BUILD FAILED")
else:
    print("node/npm missing — skipped")

In [ ]:
import shutil as _sh
PLUGINS_DIR = Path.home()/"fiftyone"/"__plugins__"/"nuscenes-trajectory"
if (PLUGIN_SRC/"dist"/"index.umd.js").exists():
    PLUGINS_DIR.mkdir(parents=True, exist_ok=True)
    for item in ["fiftyone.yml","package.json","__init__.py","dist"]:
        src=PLUGIN_SRC/item; dst=PLUGINS_DIR/item
        if dst.exists(): (_sh.rmtree(dst) if dst.is_dir() else dst.unlink())
        _sh.copytree(src,dst) if src.is_dir() else _sh.copy2(src,dst)
    print("installed to", PLUGINS_DIR, "— RESTART kernel/App (registers the operator) + hard-refresh, then add the 'Frame + Map' panel")
else:
    print("no build to install")

## 8. Use it

1. Relaunch the App on the grouped view (Section 6 cell).
2. Open a group in the modal — the App shows the **carousel** on top.
3. Add the **Frame + Map** panel (`+` menu). Use the split-pane icon to dock it **below** the Sample view so the carousel sits above it.
4. Hit **Play**: the left image steps through the scene's keyframes while the marker glides the route in sync. The status line shows `frame N / total`.
5. Click a different camera in the carousel (CAM_FRONT → CAM_BACK_LEFT → …) — the panel reloads that camera's frames; the route/marker stay anchored to the same GPS.

### Tuning & graceful degradation
- **Playback pace:** `FRAME_MS` in the panel (default 120ms/keyframe). Lower = faster.
- **Frames + marker stepping together is now the core** — driven by the operator's ordered `frames` list, no dependency on the App's group-navigation API.
- **Slice-following is best-effort** (`sample.group.name`). If your build exposes the active slice differently, it falls back to CAM_FRONT; you still get a full playable scene + map. Log the sample object in the console to adjust if needed.
- **RESTART the kernel** after install so the Python operator registers.
- **Status line** surfaces each stage (`loading scene frames…`, `operator: …`, `image failed: …`) so failures are visible, not silent.
- **LIDAR `.pcd.bin`** won't play as an image frame — the playback is meant for camera slices; selecting the lidar slice just won't produce image frames. Cameras are the show.

### Detections & map overlays
- **Camera boxes:** toggle the `cuboids` field in the App sidebar to show/hide the projected 3D→ 2D boxes on each camera. (Projection accuracy depends on nuScenes' 2D localization; some boxes near image edges may look slightly off — expected.)
- **LIDAR boxes:** the `detections` field on the LIDAR slice renders as 3D cuboids in the point-cloud viewer.
- **Crosswalks (blue) + lanes (gray)** are drawn on the Frame + Map panel from `dataset.info["map_overlays"]`, built in Section 5b. If they don't appear, the overlay build likely found no polygons in the route bbox — widen `pad` in Section 5b.
- The map-overlay build (5b) loads the city map per scene and can take a few seconds; it only converts polygons near each route, not the whole city.

### Boxes on the panel image (new)
- The panel's left image now draws the cuboids as an **SVG overlay with class-label text** (car, pedestrian, ...), stepping with Play. These come from the `cuboids` polylines the operator returns per keyframe.
- Sidebar `detections: 0` on a **camera** slice is expected — camera boxes live in the `cuboids` field; `detections` is only populated on the **LIDAR** slice (3D cuboids). Select the LIDAR slice to see those.
- If a label color/position looks slightly off at the image edge, that's the 2D-localization projection error inherent to nuScenes, not the overlay math.

### Object detections on the map (new)
- Every annotated object per keyframe is converted to **global lat/lon** (annotations are stored in nuScenes' global frame, fed through `derive_latlon`) during the Section 4 build and stashed in `dataset.info["scene_detections"]`.
- The map shows the **whole scene's objects as faint dots** plus the **current keyframe's objects as bright outlined dots**, color-coded by class, with a **legend** (top-right of the map).
- As you Play, the bright dots update per keyframe in sync with the ego marker — a live bird's-eye view of the scene's objects on the real map.
- All classes are included. Requires rebuilding the dataset (Section 4) so the detection lat/lons get computed and stored.

### LIDAR carousel thumbnails (Section 5c)
- The broken-file icon on the LIDAR carousel slice is the App trying to render raw `.pcd.bin`. Section 5c fixes it for real: converts each cloud to `.pcd`, wraps it in a `.fo3d` scene, repoints the sample, and computes **orthographic projection** thumbnails. After running 5c + relaunch, the carousel shows a top-down BEV preview.
- Run 5c **after** the build (Section 4) and **before** launching. It writes derived files under `~/datasets/nuscenes_derived/` and is heavier (one `.pcd` + `.fo3d` per keyframe).
- This also makes the LIDAR slice render as a real 3D point cloud in the App's modal viewer (not just our panel's vector BEV).
- Our panel's vector BEV (oriented boxes) is independent of this — it reads `detections`, not the filepath — so both work after 5c.